Import necessary libraries

In [ ]:
import polars as pl
import numpy as np
from scipy import stats

## 1. Hypothesis testing

In [ ]:
df = pl.scan_csv("../datasets/spotify_tracks_preprocessed.csv")

#### 1.1 Are songs released in the "Streaming Era" (2015–2023) are significantly shorter than songs released in the "Digital Download Era" (2000–2010)?

This hypothesis comes from insight that streaming platforms pay per play (after 30 seconds). Therefore, shorter songs allow for more plays per hour, incentivizing artists to shorten tracks. We are comparing the "Streaming Era" (2015–2023) against the "Digital Download/iTunes Era" (2000–2010).

*   **Statistical Test:** **Two-Sample t-test** (Comparing means of two independent groups with normal distributions).
*   **Null Hypothesis ($H_0$):** There is no difference in the mean duration of songs between the 2000–2010 period and the 2015–2023 period.
*   **Observation:** Chart _2.3.2 Long Song_ showed a steep linear decline in average song duration starting around 2010, dropping from ~4.3 minutes to ~3.4 minutes.


In [ ]:
streaming_era = (
    df.filter(pl.col("year").is_between(2015, 2023))
    .select("duration_ms")
    .collect()
    .to_numpy()
    .squeeze()
)
digital_era = (
    df.filter(pl.col("year").is_between(2000, 2010))
    .select("duration_ms")
    .collect()
    .to_numpy()
    .squeeze()
)

# assumes unequal variance
t_stat, p_val = stats.ttest_ind(
    streaming_era,
    digital_era,
    equal_var=False,
    alternative="less",
)

print(
    f"1. Streaming Era Mean (ms): {streaming_era.mean():.2f} ~ {streaming_era.mean() / 60000:.2f} mins"
)
print(
    f"2. Digital Era Mean (ms):   {digital_era.mean():.2f} ~ {digital_era.mean() / 60000:.2f} mins"
)
print(f"3. P-value: {p_val}")

**Result:**
We **Reject the Null Hypothesis** ($p \approx 0.0$).

**Interpretation:**
There is a statistically significant difference in song duration between the two eras. Songs released in the **Streaming Era (2015–2023)** have an average length of **3 minutes and 42 seconds** (222,586ms), which is approximately **37 seconds shorter** than songs from the **Digital Download Era (2000–2010)**, which averaged **4 minutes and 20 seconds** (259,982ms).

This strongly supports the theory that the streaming economy, which pays per play, has created a financial incentive for artists to write shorter songs to increase the frequency of plays.

#### 1.2 Is the median popularity of "Hip-Hop & R&B" tracks statistically higher than the median popularity of "Rock & Metal" tracks in the 2023 ecosystem?

In the 2023 ecosystem, "Hip-Hop" and "R&B" are hypothesized to be the dominant cultural forces, driving higher engagement (popularity) than the previous giants of "Rock" and "Metal." Since popularity is not always a normal bell curve, we check the median (the middle value) using the Mann-Whitney U test.

*   **Statistical Test:** **Mann-Whitney U Test** (Comparing two independent groups with non-normal distributions).
*   **Null Hypothesis ($H_0$):** The median popularity distribution of Hip-Hop & R&B is equal to that of Rock & Metal.
*   **Observation:** Chart _2.4.1 Popularity by Super Genre_ revealed that Hip-Hop & R&B had the highest median popularity (25), while Rock & Metal was significantly lower (16).

In [ ]:
hh_rnb_pop = (
    df.filter(pl.col("genre").str.contains("(?i)Hip-Hop|R&B|Rap"))
    .select("popularity")
    .collect()
    .to_numpy()
    .squeeze()
)
rock_metal_pop = (
    df.filter(pl.col("genre").str.contains("(?i)Rock|Metal"))
    .select("popularity")
    .collect()
    .to_numpy()
    .squeeze()
)

# alternative='greater' tests if hh_rnb_pop median > rock_metal_pop median
u_stat, p_val = stats.mannwhitneyu(hh_rnb_pop, rock_metal_pop, alternative="greater")

print(f"1. Median Popularity (Hip-Hop/R&B): {np.median(hh_rnb_pop)}")
print(f"2. Median Popularity (Rock/Metal):  {np.median(rock_metal_pop)}")
print(f"3. P-value: {p_val}")

**Result:**
We **Reject the Null Hypothesis** ($p \approx 0.0$).

**Interpretation:**
The data reveals a massive "Genre Gap" in the current music ecosystem. Tracks categorized as **Hip-Hop & R&B** have a median popularity score of **37**, which is **more than double** the median popularity of **Rock & Metal** (17).

This statistically confirms that while Rock & Metal have dedicated fanbases, the general "center of gravity" for music consumption and engagement has shifted decisively toward Hip-Hop and R&B culture.

#### 1.3 Do tracks classified as **"Vocal-Centric"** (`instrumentalness` < 0.1) have a statistically higher median popularity than **"Instrumental"** tracks (`instrumentalness` > 0.5)?

**The Theory:**
Despite the rise of "Lo-Fi Study Beats" and functional playlists, this hypothesis suggests that the music market still heavily favors human voices. We are testing if "Vocal-Centric" songs (where `instrumentalness` is near 0) are statistically more popular than "Instrumental" tracks (where `instrumentalness` is high).

*   **Statistical Test:** **Mann-Whitney U Test** (Comparing two independent groups with non-normal distributions).
*   **Null Hypothesis ($H_0$):** There is no difference in the median popularity between vocal-centric tracks and instrumental tracks.
*   **Observation:** This validates the "Personality" aspect of the music business. If rejected, it proves that "Background Music" (utility listening) has become just as commercially viable in the streaming era as traditional "Foreground Music" (active listening).

In [ ]:
vocal_tracks = (
    df.filter(pl.col("instrumentalness") < 0.1)
    .select("popularity")
    .collect()
    .to_numpy()
    .squeeze()
)
inst_tracks = (
    df.filter(pl.col("instrumentalness") > 0.5)
    .select("popularity")
    .collect()
    .to_numpy()
    .squeeze()
)

# alternative='greater' checks if Vocal is higher
u_stat, p_val = stats.mannwhitneyu(vocal_tracks, inst_tracks, alternative="greater")

print(f"1. Median Popularity (Vocal): {np.median(vocal_tracks)}")
print(f"2. Median Popularity (Instrumental): {np.median(inst_tracks)}")
print(f"3. P-value: {p_val}")

**Result:**
We **Reject the Null Hypothesis** ($p \approx 0.0$).

**Interpretation:**
The data supports the "Vocal Economy" theory. Vocal-centric tracks have a **median popularity of 15**, which is significantly higher than Instrumental tracks (median popularity of 9).

While instrumental music (like Lo-Fi or Classical) has a niche, the broader market clearly prefers tracks with vocal content. The "human element" remains a primary driver of listener engagement.

#### 1.4 Do tracks falling into the "Rhythmic/Rap" speechiness range ($0.33 - 0.66$) have a higher mean popularity than tracks in the "Melodic/Music" range ($< 0.33$)?

This hypothesis explores the specific impact of lyrical density. We are testing if tracks that mix speech and music (the "Rap/Rhythmic" zone, `speechiness` 0.33–0.66) are more successful on average than standard "Melodic" tracks (`speechiness` < 0.33).

*   **Statistical Test:** **Welch’s t-test** (assuming unequal variances/sample sizes between the massive "Music" group and the "Rap" group).
*   **Null Hypothesis ($H_0$):** The mean popularity of "Rhythmic/Rap" tracks is less than or equal to that of "Melodic/Music" tracks.
*   **Observation:** This quantifies the dominance of Hip-Hop culture. If true, it suggests that adding rhythmic speech elements to a track is currently the most effective way to increase its commercial viability, more so than traditional singing.

In [ ]:
rap_zone = (
    df.filter(pl.col("speechiness").is_between(0.33, 0.66))
    .select("popularity")
    .collect()
    .to_numpy()
    .squeeze()
)
music_zone = (
    df.filter(pl.col("speechiness") < 0.33)
    .select("popularity")
    .collect()
    .to_numpy()
    .squeeze()
)

# alternative='greater' checks if Rap Zone is higher than Music Zone
t_stat, p_val = stats.ttest_ind(
    rap_zone, music_zone, equal_var=False, alternative="greater"
)

print(f"1. Mean Popularity (Rap Zone):   {rap_zone.mean():.2f}")
print(f"2. Mean Popularity (Melodic Zone): {music_zone.mean():.2f}")
print(f"3. P-value: {p_val}")

**Result:**
We **Fail to Reject the Null Hypothesis** ($p \approx 0.29$).

**Interpretation:**
Unlike the Genre test (Hypothesis 2), which showed Hip-Hop dominates Rock, the specific **audio feature** of `speechiness` does not predict popularity on its own.
The mean popularity of "Rap/Rhythmic" tracks (17.02) is virtually identical to "Melodic" tracks (16.98).

**Insight:** This suggests that while the *culture* of Hip-Hop is popular, the technical density of words (speechiness > 0.33) is not the deciding factor. A melodic Pop song is just as likely to be a hit as a lyrical Rap song.

#### 1.5 Is there a non-linear (quadratic) relationship between Duration and Popularity, where popularity peaks for songs between **2:30 (150,000ms)** and **3:30 (210,000ms)** and significantly decreases for songs shorter than 2:00 or longer than 4:00?

Hypothesis 1 proved songs are getting shorter, but this hypothesis asks: **Is there a limit?** We are testing if popularity peaks in a specific "Sweet Spot" (2:30 to 3:30) compared to songs that are "Too Short" (< 2:00) or "Too Long" (> 4:00).

*   **Statistical Test:** **ANOVA** (by binning duration into Short/Optimal/Long).
*   **Null Hypothesis ($H_0$):** Popularity is distributed evenly across all duration bins, or the relationship is strictly linear.
*   **Observation:** This detects "Optimization" in the streaming economy. If the peak is strictly between 2:30–3:30, it confirms that artists are engineering song lengths to maximize stream counts without making the song so short it feels incomplete to the listener.

In [ ]:
# Define bins in milliseconds
# Short: < 2:00 (120,000ms)
# Optimal: 2:30 - 3:30 (150,000ms - 210,000ms)
# Long: > 4:00 (240,000ms)

short_group = (
    df.filter(pl.col("duration_ms") < 120_000)
    .select("popularity")
    .collect()
    .to_numpy()
    .squeeze()
)
optimal_group = (
    df.filter(pl.col("duration_ms").is_between(150_000, 210_000))
    .select("popularity")
    .collect()
    .to_numpy()
    .squeeze()
)
long_group = (
    df.filter(pl.col("duration_ms") > 240_000)
    .select("popularity")
    .collect()
    .to_numpy()
    .squeeze()
)

# ANOVA Test (Analysis of Variance)
# Tests if there is ANY statistically significant difference between the three groups
f_stat, p_val = stats.f_oneway(short_group, optimal_group, long_group)

print(f"1. Mean Pop (Short < 2m): {short_group.mean():.2f}")
print(f"2. Mean Pop (Optimal 2.5-3.5m): {optimal_group.mean():.2f}")
print(f"3. Mean Pop (Long > 4m): {long_group.mean():.2f}")
print(f"4. P-value: {p_val}")

**Result:**
We **Reject the Null Hypothesis** ($p \approx 0.0$).

**Interpretation:**
The data confirms a distinct "Attention Economy Optimization." Popularity does **not** simply increase as songs get shorter; instead, it follows a non-linear "inverted-U" curve:
*   **The "Sweet Spot" (2:30 – 3:30):** This range performs best, with a mean popularity of **18.95**.
*   **Too Long (> 4:00):** Popularity drops to **15.21**.
*   **Too Short (< 2:00):** Popularity is actually the lowest at **12.95**.

**Insight:** This suggests that while streaming incentivizes shorter songs (Hypothesis 1), there is a "floor." Tracks under 2 minutes (often interludes or unfinished ideas) fail to gain traction. The industry has effectively optimized the "Perfect Pop Song" length to be roughly **3 minutes**—long enough to satisfy the listener, but short enough to maximize play counts.

### Final Data Story
If you were to engineer the statistically "perfect" song for the 2023 streaming economy, the data suggests it should be:
> A **Hip-Hop or R&B** track that focuses on **Vocals** (not instrumental beats), lasting exactly **2 minutes and 50 seconds**. While the genre matters, you don't necessarily need to rap (high speechiness)—a melodic hook works just as well.

## 2. Unsupervised Learning for Genre Discovery

In this section, we will apply K-Means Clustering to identify natural groupings of songs based on their audio features, potentially revealing new or hybrid genres that are not captured by traditional labels.

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

import plotly.express as px
import plotly.graph_objects as go

#### 2.1 Data Preparation

In [ ]:
features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
]

pdf = df.select(features + ["track_name", "artist_name"]).collect().to_pandas().dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pdf[features])

#### 2.2 Clustering

In [ ]:
# we try first with 5 clusters
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# add labels back to the dataframe
pdf["Cluster"] = cluster_labels
pdf["Cluster_Label"] = pdf["Cluster"].astype(str)

#### 2.3 Visualization

In [ ]:
# Reduce 9 dimensions down to 2 so we can plot it on a screen
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)
pdf["PCA1"] = components[:, 0]
pdf["PCA2"] = components[:, 1]

In [ ]:
fig_pca = px.scatter(
    pdf,
    x="PCA1",
    y="PCA2",
    color="Cluster_Label",
    # NOW this will work because 'track_name' exists in pdf
    hover_data=["track_name", "artist_name"],
    title="The 'Map' of Music: 5 Audio Clusters",
    color_discrete_sequence=px.colors.qualitative.Bold,
    opacity=0.7,
    width=900,
    height=600,
)
fig_pca.update_traces(marker=dict(size=6))
fig_pca.show()

Visual Analysis of Clusters:

1.  **The "Outlier" (Cluster 3 - Yellow):**
    *   The Yellow cluster stretches far out to the left, completely separating itself from the main blob.
    *   **Interpretation:** In music PCA, the X-axis (horizontal) usually represents **Energy vs. Acousticness**. This Yellow group is likely **Acoustic / Instrumental / Low-Energy** tracks (e.g., Classical, Jazz, or Ballads) that are structurally different from everything else.

2.  **The "Mainstream Core" (Red, Blue, Green):**
    *   Clusters 4 (Red), 2 (Blue), and 0 (Green) are clumped together on the right but occupy different "territories" vertically.
    *   **Interpretation:** These likely represent the variations of modern produced music (Pop, Hip-Hop, Rock). They share high production values (loudness/energy) but differ in **Mood (Valence)** or **Rhythm (Danceability)**.

In [ ]:
cluster_means = pdf.groupby("Cluster")[features].mean()

# scale to 0-1 for the chart
min_max_scaler = MinMaxScaler()
cluster_means_norm = pd.DataFrame(
    min_max_scaler.fit_transform(cluster_means),
    columns=features,
    index=cluster_means.index,
)

fig_radar = go.Figure()
colors = px.colors.qualitative.Bold

for i in range(len(cluster_means_norm)):
    fig_radar.add_trace(
        go.Scatterpolar(
            r=cluster_means_norm.iloc[i].values,
            theta=features,
            fill="toself",
            name=f"Cluster {i}",
            line_color=colors[i % len(colors)],
            opacity=0.6,
        )
    )

fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Audio Personalities (Cluster Signatures)",
    width=800,
    height=600,
)
fig_radar.show()

1. Cluster 3 (Yellow): "The Chill Zone" (Acoustic/Ambient)
   *   **Signature:** Massive spikes in **Acousticness** and **Instrumentalness**. Very low Energy and Loudness.
   *   **Explanation:** This is the "Outlier" we saw in the PCA plot (the separated yellow blob). These are likely Lo-Fi beats, Classical tracks, or acoustic ballads.

2. Cluster 4 (Red): "The Lyrical Flow" (Rap/Hip-Hop)
   *   **Signature:** The only cluster with a huge spike in **Speechiness**.
   *   **Explanation:** This represents the "Rhythmic/Rap" tracks. The model successfully separated these based purely on the density of spoken words.

3. Cluster 1 (Green): "High Octane" (Rock/Metal/High-Energy)
   *   **Signature:** Dominates in **Loudness**, **Energy**, and **Tempo**.
   *   **Explanation:** These are the aggressive tracks. If you listen to this cluster, you'll likely hear distorted guitars or heavy EDM beats.

4. Cluster 2 (Blue): "Good Vibes" (Happy Pop)
   *   **Signature:** High **Valence** (Happiness) and **Danceability**.
   *   **Explanation:** This is the "Radio Friendly" cluster. It’s designed to make you move and feel good. It sits right in the middle of the energy spectrum—not too aggressive (Green), not too sleepy (Yellow).

5. Cluster 0 (Purple): "The Live Experience"
   *   **Signature:** A solitary, massive spike in **Liveness**.
   *   **Explanation:** The model detected the specific audio artifacts of live audiences (cheering, reverb). These tracks are structurally different from studio recordings.